# Phase 0b: Préparation du Dataset Kaggle-Only (100% Kaggle)

## Contexte

Le Stage 1 original entraîné sur données mixtes (60% Kaggle + 40% externes) a obtenu:
- **Validation**: 80.5%
- **Test (Kaggle-only)**: **15.5%** ❌

**Root Cause**: Distribution shift massif car test set = 100% Kaggle mais training = sources multiples.

## Solution

Utiliser **uniquement** les données Kaggle (21,397 images) avec équilibrage par augmentation:
- Cible: **4,000 images par classe** (20,000 total)
- Split: **80/20** train/val stratifié
- Output: 16,000 train + 4,000 val

## Distribution Naturelle Kaggle

| Classe | Count | % | Action |
|--------|-------|---|--------|
| CMD | 13,158 | 61.5% | Sous-échantillonner → 4,000 |
| Healthy | 2,577 | 12.0% | Augmenter 1.5x → 4,000 |
| CGM | 2,386 | 11.2% | Augmenter 1.7x → 4,000 |
| CBSD | 2,189 | 10.2% | Augmenter 1.8x → 4,000 |
| CBB | 1,087 | 5.1% | Augmenter 3.7x → 4,000 |

## 1. Imports et Configuration

In [ ]:
import os
import json
import random
from pathlib import Path
from collections import Counter, defaultdict
from typing import List, Dict, Tuple

import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Installation albumentations si nécessaire
try:
    import albumentations as A
    from albumentations import (
        Compose, HorizontalFlip, VerticalFlip, Rotate,
        RandomBrightnessContrast, HueSaturationValue,
        GaussianBlur, CLAHE, ShiftScaleRotate
    )
except ImportError:
    print("Installing albumentations...")
    !pip install -q albumentations
    import albumentations as A
    from albumentations import (
        Compose, HorizontalFlip, VerticalFlip, Rotate,
        RandomBrightnessContrast, HueSaturationValue,
        GaussianBlur, CLAHE, ShiftScaleRotate
    )

# Configuration
BASE_DIR = Path("/home/hounfodjidagba/learning/cassava")
DATA_DIR = BASE_DIR / "data"
KAGGLE_IMAGES_DIR = DATA_DIR / "raw" / "train_images"
KAGGLE_CSV = DATA_DIR / "raw" / "train.csv"
OUTPUT_DIR = BASE_DIR / "new_implementation" / "data" / "kaggle_only"
AUGMENTED_DIR = OUTPUT_DIR / "augmented_images"

# Créer dossiers de sortie
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
AUGMENTED_DIR.mkdir(parents=True, exist_ok=True)

# Mapping des labels
LABEL_TO_CLASS = {
    0: "CBB",
    1: "CBSD",
    2: "CGM",
    3: "CMD",
    4: "Healthy"
}

CLASS_TO_LABEL = {v: k for k, v in LABEL_TO_CLASS.items()}

# Noms complets pour les réponses
LABEL_TO_FULL_NAME = {
    0: "Cassava Bacterial Blight (CBB)",
    1: "Cassava Brown Streak Disease (CBSD)",
    2: "Cassava Green Mottle (CGM)",
    3: "Cassava Mosaic Disease (CMD)",
    4: "Healthy"
}

# Configuration équilibrage
TARGET_PER_CLASS = 4000
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"Base directory: {BASE_DIR}")
print(f"Kaggle images: {KAGGLE_IMAGES_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Target per class: {TARGET_PER_CLASS}")

## 2. Charger et Analyser les Données Kaggle

In [ ]:
print("=== Chargement des données Kaggle ===")
print()

# Charger labels
df_kaggle = pd.read_csv(KAGGLE_CSV)
print(f"Total samples dans CSV: {len(df_kaggle)}")

# Vérifier existence des images
kaggle_samples = []
missing_count = 0

for idx, row in tqdm(df_kaggle.iterrows(), total=len(df_kaggle), desc="Vérification images"):
    image_path = KAGGLE_IMAGES_DIR / row['image_id']
    
    if not image_path.exists():
        missing_count += 1
        continue
    
    kaggle_samples.append({
        'image_id': row['image_id'],
        'image_path': str(image_path.absolute()),
        'label': int(row['label']),
        'class_name': LABEL_TO_CLASS[int(row['label'])]
    })

print(f"\n✓ Images valides: {len(kaggle_samples)}")
print(f"✗ Images manquantes: {missing_count}")

In [ ]:
# Distribution des classes
print("\n=== Distribution Naturelle Kaggle ===")
print()

samples_by_class = defaultdict(list)
for sample in kaggle_samples:
    samples_by_class[sample['class_name']].append(sample)

total = len(kaggle_samples)
class_counts = {}

for class_name in ["CBB", "CBSD", "CGM", "CMD", "Healthy"]:
    count = len(samples_by_class[class_name])
    class_counts[class_name] = count
    pct = count / total * 100
    action = "Sous-échantillonner" if count > TARGET_PER_CLASS else f"Augmenter {TARGET_PER_CLASS/count:.1f}x"
    print(f"{class_name:8s}: {count:6d} ({pct:5.1f}%) → {action}")

print(f"\nTotal: {total}")

In [ ]:
# Visualisation distribution originale
fig, ax = plt.subplots(figsize=(10, 6))

classes = ["CBB", "CBSD", "CGM", "CMD", "Healthy"]
counts = [class_counts[c] for c in classes]
colors = ['#ff6b6b', '#feca57', '#48dbfb', '#ff9ff3', '#1dd1a1']

bars = ax.bar(classes, counts, color=colors, edgecolor='black')
ax.axhline(y=TARGET_PER_CLASS, color='red', linestyle='--', label=f'Cible: {TARGET_PER_CLASS}')

ax.set_ylabel('Nombre d\'images', fontsize=12)
ax.set_title('Distribution Naturelle Kaggle (avant équilibrage)', fontsize=14, fontweight='bold')
ax.legend()

for bar, count in zip(bars, counts):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 100,
            f'{count}\n({count/total*100:.1f}%)',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'kaggle_distribution_before.png', dpi=150)
plt.show()

print(f"✓ Graphique sauvegardé: {OUTPUT_DIR / 'kaggle_distribution_before.png'}")

## 3. Définir Augmentation Conservative

In [ ]:
# Augmentation conservative pour garder le réalisme
# Objectif: créer des variations subtiles, pas des images irréalistes

augmentation_transform = Compose([
    # Rotation légère (feuilles peuvent être orientées différemment)
    Rotate(limit=15, p=0.5, border_mode=0),
    
    # Flips (feuilles peuvent être vues de n'importe quel côté)
    HorizontalFlip(p=0.5),
    VerticalFlip(p=0.3),
    
    # Variations de luminosité/contraste (conditions d'éclairage différentes)
    RandomBrightnessContrast(
        brightness_limit=0.15,  # Conservative
        contrast_limit=0.15,
        p=0.5
    ),
    
    # Légères variations de couleur (différentes conditions de capture)
    HueSaturationValue(
        hue_shift_limit=8,       # Très conservatif
        sat_shift_limit=12,
        val_shift_limit=10,
        p=0.3
    ),
    
    # Léger flou (focus variable)
    GaussianBlur(blur_limit=(3, 5), p=0.15),
])

print("✓ Augmentation transform configuré (conservative)")
print("\nTransforms:")
print("  - Rotation: ±15° (p=0.5)")
print("  - HorizontalFlip (p=0.5)")
print("  - VerticalFlip (p=0.3)")
print("  - Brightness/Contrast: ±15% (p=0.5)")
print("  - Hue/Sat/Val: ±8/±12/±10 (p=0.3)")
print("  - GaussianBlur: 3-5px (p=0.15)")

## 4. Fonction d'Augmentation

In [ ]:
def augment_to_target(samples: List[Dict], target_count: int, class_name: str) -> List[Dict]:
    """
    Augmente une classe jusqu'à target_count.
    
    Args:
        samples: Liste des échantillons originaux de la classe
        target_count: Nombre cible d'échantillons
        class_name: Nom de la classe (pour nommage fichiers)
    
    Returns:
        Liste augmentée (originaux + augmentés)
    """
    current_count = len(samples)
    
    if current_count >= target_count:
        # Sous-échantillonner
        return random.sample(samples, target_count)
    
    # Besoin d'augmenter
    needed = target_count - current_count
    print(f"  {class_name}: {current_count} → {target_count} (besoin de {needed} augmentés)")
    
    # Garder tous les originaux
    result = samples.copy()
    
    # Créer dossier pour classe
    class_aug_dir = AUGMENTED_DIR / class_name.lower()
    class_aug_dir.mkdir(parents=True, exist_ok=True)
    
    # Générer augmentations
    aug_idx = 0
    pbar = tqdm(total=needed, desc=f"    Augmenting {class_name}")
    
    while len(result) < target_count:
        # Choisir image source aléatoire
        source_sample = random.choice(samples)
        
        try:
            # Charger image
            img = Image.open(source_sample['image_path'])
            img_array = np.array(img)
            
            # Appliquer augmentation
            augmented = augmentation_transform(image=img_array)
            aug_image = augmented['image']
            
            # Sauvegarder
            aug_filename = f"{class_name.lower()}_aug_{aug_idx:05d}.jpg"
            aug_path = class_aug_dir / aug_filename
            
            Image.fromarray(aug_image).save(aug_path, quality=95)
            
            # Ajouter au résultat
            result.append({
                'image_id': aug_filename,
                'image_path': str(aug_path.absolute()),
                'label': source_sample['label'],
                'class_name': source_sample['class_name'],
                'is_augmented': True,
                'source_image': source_sample['image_id']
            })
            
            aug_idx += 1
            pbar.update(1)
            
        except Exception as e:
            print(f"\n    Erreur augmentation: {e}")
            continue
    
    pbar.close()
    return result

print("✓ Fonction d'augmentation définie")

## 5. Équilibrer les Classes

In [ ]:
print("=== Équilibrage des Classes ===")
print(f"Cible par classe: {TARGET_PER_CLASS}")
print()

balanced_samples = {}

for class_name in ["CBB", "CBSD", "CGM", "CMD", "Healthy"]:
    original_samples = samples_by_class[class_name]
    balanced = augment_to_target(original_samples, TARGET_PER_CLASS, class_name)
    balanced_samples[class_name] = balanced
    print()

# Fusionner toutes les classes
all_balanced = []
for class_name, samples in balanced_samples.items():
    all_balanced.extend(samples)

print(f"\n✓ Total équilibré: {len(all_balanced)} images")
print(f"  ({len(all_balanced) // 5} images × 5 classes)")

In [ ]:
# Vérifier distribution équilibrée
print("\n=== Distribution Après Équilibrage ===")
print()

balanced_counts = Counter([s['class_name'] for s in all_balanced])
total_balanced = len(all_balanced)

for class_name in ["CBB", "CBSD", "CGM", "CMD", "Healthy"]:
    count = balanced_counts[class_name]
    orig = class_counts[class_name]
    aug_count = sum(1 for s in all_balanced if s['class_name'] == class_name and s.get('is_augmented', False))
    print(f"{class_name:8s}: {count:5d} (orig: {orig:5d}, aug: {aug_count:5d}) - {count/total_balanced*100:.1f}%")

print(f"\nTotal: {total_balanced}")

In [ ]:
# Visualisation comparaison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

classes = ["CBB", "CBSD", "CGM", "CMD", "Healthy"]
colors = ['#ff6b6b', '#feca57', '#48dbfb', '#ff9ff3', '#1dd1a1']

# Avant
counts_before = [class_counts[c] for c in classes]
bars1 = ax1.bar(classes, counts_before, color=colors, edgecolor='black', alpha=0.8)
ax1.axhline(y=TARGET_PER_CLASS, color='red', linestyle='--', label=f'Cible')
ax1.set_ylabel('Nombre d\'images')
ax1.set_title('AVANT Équilibrage', fontweight='bold')
ax1.legend()
for bar, count in zip(bars1, counts_before):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 100,
            str(count), ha='center', fontsize=10, fontweight='bold')

# Après
counts_after = [balanced_counts[c] for c in classes]
bars2 = ax2.bar(classes, counts_after, color=colors, edgecolor='black', alpha=0.8)
ax2.axhline(y=TARGET_PER_CLASS, color='red', linestyle='--', label=f'Cible')
ax2.set_ylabel('Nombre d\'images')
ax2.set_title('APRÈS Équilibrage', fontweight='bold')
ax2.legend()
for bar, count in zip(bars2, counts_after):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
            str(count), ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'kaggle_distribution_comparison.png', dpi=150)
plt.show()

print(f"✓ Graphique sauvegardé: {OUTPUT_DIR / 'kaggle_distribution_comparison.png'}")

## 6. Split Stratifié Train/Val (80/20)

In [ ]:
print("=== Split Stratifié Train/Val ===")
print("Split: 80% train / 20% val")
print()

# Préparer pour stratification
X = all_balanced
y = [s['label'] for s in all_balanced]

# Split stratifié
train_samples, val_samples = train_test_split(
    X, 
    test_size=0.2, 
    stratify=y, 
    random_state=RANDOM_SEED
)

print(f"Train: {len(train_samples)} ({len(train_samples)/len(all_balanced)*100:.0f}%)")
print(f"Val:   {len(val_samples)} ({len(val_samples)/len(all_balanced)*100:.0f}%)")
print()

# Vérifier stratification
print("Vérification stratification:")
train_dist = Counter([s['class_name'] for s in train_samples])
val_dist = Counter([s['class_name'] for s in val_samples])

print(f"{'Classe':8s} | {'Train':>6s} | {'Val':>6s} | {'Train %':>8s} | {'Val %':>8s}")
print("-" * 50)
for class_name in classes:
    t = train_dist[class_name]
    v = val_dist[class_name]
    print(f"{class_name:8s} | {t:6d} | {v:6d} | {t/len(train_samples)*100:7.1f}% | {v/len(val_samples)*100:7.1f}%")

## 7. Convertir au Format Qwen

In [ ]:
def convert_to_qwen_format(samples: List[Dict]) -> List[Dict]:
    """
    Convertit les échantillons au format conversation Qwen.
    
    Format de réponse: "Disease: [CLASS_NAME]" (simple et discriminatif)
    """
    qwen_samples = []
    
    for sample in samples:
        # Utiliser le nom complet pour la réponse
        full_name = LABEL_TO_FULL_NAME[sample['label']]
        
        qwen_entry = {
            "id": sample['image_id'],
            "conversations": [
                {
                    "from": "user",
                    "value": f"Picture 1: <img>{sample['image_path']}</img>\nWhat disease is affecting this cassava plant?"
                },
                {
                    "from": "assistant",
                    "value": f"Disease: {full_name}"
                }
            ],
            "metadata": {
                "class_name": sample['class_name'],
                "label": sample['label'],
                "source": "kaggle_competition",
                "is_augmented": sample.get('is_augmented', False)
            }
        }
        qwen_samples.append(qwen_entry)
    
    return qwen_samples

print("\n=== Conversion au Format Qwen ===")
print()

train_qwen = convert_to_qwen_format(train_samples)
val_qwen = convert_to_qwen_format(val_samples)

print(f"✓ Train: {len(train_qwen)} conversations")
print(f"✓ Val:   {len(val_qwen)} conversations")

In [ ]:
# Aperçu format
print("\n=== Aperçu du Format ===")
print()

sample = random.choice(train_qwen)
print(f"ID: {sample['id']}")
print(f"Class: {sample['metadata']['class_name']}")
print(f"Augmented: {sample['metadata']['is_augmented']}")
print()
print("User:")
user_msg = sample['conversations'][0]['value']
# Afficher avec chemin tronqué
img_part = user_msg.split('<img>')[1].split('</img>')[0]
print(f"  Picture 1: <img>.../{Path(img_part).name}</img>")
print(f"  {user_msg.split('</img>')[1].strip()}")
print()
print("Assistant:")
print(f"  {sample['conversations'][1]['value']}")

## 8. Sauvegarder les Datasets

In [ ]:
print("=== Sauvegarde des Datasets ===")
print()

# Sauvegarder train
train_file = OUTPUT_DIR / "cassava_train.json"
with open(train_file, 'w', encoding='utf-8') as f:
    json.dump(train_qwen, f, ensure_ascii=False, indent=2)
print(f"✓ Train: {train_file}")
print(f"  → {len(train_qwen)} échantillons")

# Sauvegarder val
val_file = OUTPUT_DIR / "cassava_val.json"
with open(val_file, 'w', encoding='utf-8') as f:
    json.dump(val_qwen, f, ensure_ascii=False, indent=2)
print(f"✓ Val:   {val_file}")
print(f"  → {len(val_qwen)} échantillons")

# Sauvegarder statistiques
stats = {
    "phase": "0b",
    "description": "Kaggle-only balanced dataset for Stage 1 retraining",
    "source": "100% Kaggle competition data",
    "original_total": len(kaggle_samples),
    "target_per_class": TARGET_PER_CLASS,
    "total_balanced": len(all_balanced),
    "train_count": len(train_qwen),
    "val_count": len(val_qwen),
    "split_ratio": "80/20",
    "distribution_before": class_counts,
    "distribution_after": dict(balanced_counts),
    "augmented_counts": {
        cls: sum(1 for s in all_balanced if s['class_name'] == cls and s.get('is_augmented', False))
        for cls in classes
    },
    "label_mapping": LABEL_TO_CLASS,
    "label_to_full_name": LABEL_TO_FULL_NAME,
    "random_seed": RANDOM_SEED
}

stats_file = OUTPUT_DIR / "dataset_stats.json"
with open(stats_file, 'w', encoding='utf-8') as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)
print(f"✓ Stats: {stats_file}")

## 9. Vérification Qualité Augmentation

In [ ]:
# Afficher quelques exemples d'augmentation
print("=== Exemples d'Augmentation ===")
print()

# Trouver paires original/augmenté
augmented_samples = [s for s in all_balanced if s.get('is_augmented', False)]

if augmented_samples:
    # Prendre 3 exemples aléatoires
    examples = random.sample(augmented_samples, min(3, len(augmented_samples)))
    
    fig, axes = plt.subplots(len(examples), 2, figsize=(10, 4*len(examples)))
    if len(examples) == 1:
        axes = [axes]
    
    for idx, aug_sample in enumerate(examples):
        # Trouver original
        source_id = aug_sample.get('source_image')
        original = next((s for s in kaggle_samples if s['image_id'] == source_id), None)
        
        if original:
            # Charger images
            orig_img = Image.open(original['image_path'])
            aug_img = Image.open(aug_sample['image_path'])
            
            # Afficher
            axes[idx][0].imshow(orig_img)
            axes[idx][0].set_title(f"Original ({original['class_name']})")
            axes[idx][0].axis('off')
            
            axes[idx][1].imshow(aug_img)
            axes[idx][1].set_title(f"Augmenté ({aug_sample['class_name']})")
            axes[idx][1].axis('off')
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'augmentation_examples.png', dpi=150)
    plt.show()
    
    print(f"✓ Exemples sauvegardés: {OUTPUT_DIR / 'augmentation_examples.png'}")
else:
    print("Pas d'images augmentées (toutes les classes avaient assez de données)")

## 10. Résumé Final

In [ ]:
print("="*60)
print("PHASE 0b TERMINÉE: Dataset Kaggle-Only Préparé")
print("="*60)
print()
print("📊 STATISTIQUES")
print(f"  Source: 100% Kaggle competition data")
print(f"  Images originales: {len(kaggle_samples):,}")
print(f"  Cible par classe: {TARGET_PER_CLASS:,}")
print(f"  Total équilibré: {len(all_balanced):,}")
print()
print("📂 FICHIERS GÉNÉRÉS")
print(f"  Train: {OUTPUT_DIR / 'cassava_train.json'}")
print(f"         → {len(train_qwen):,} échantillons (80%)")
print(f"  Val:   {OUTPUT_DIR / 'cassava_val.json'}")
print(f"         → {len(val_qwen):,} échantillons (20%)")
print(f"  Stats: {OUTPUT_DIR / 'dataset_stats.json'}")
print()
print("📈 DISTRIBUTION ÉQUILIBRÉE")
for class_name in classes:
    count = balanced_counts[class_name]
    print(f"  {class_name:8s}: {count:5,} ({count/total_balanced*100:.1f}%)")
print()
print("✅ PROCHAINE ÉTAPE")
print("  Phase 0c: Créer notebook 17_train_stage1_kaggle_only.ipynb")
print("  pour entraîner Stage 1 sur ce dataset")
print()
print("="*60)